# 🤖 Agentic RAG with LangGraph

**Models:** `gpt-4o-mini` (agents) · `text-embedding-3-small` (embeddings)  
**Vector store:** FAISS (in-memory, no server needed)  
**Framework:** LangGraph 1.2.6 · LangChain 1.3.x  

### Architecture
```
User Question
      │
      ▼
 ┌────────────────┐
 │  Orchestrator  │◄──────────────────────┐
 │     Agent      │                        │
 └───────┬────────┘                        │
         │ route                           │
    ┌────┴────┐                            │
    ▼         ▼                            │
 ┌──────┐  ┌──────────┐                   │
 │ RAG  │  │Calculator│  ← Tools          │
 │Search│  │Summarise │                   │
 └──┬───┘  └────┬─────┘                   │
    └─────┬─────┘                         │
          ▼                               │
   ┌─────────────┐                        │
   │  Research   │────── tool calls ──────┘
   │    Agent    │
   └──────┬──────┘
          ▼
   ┌─────────────┐
   │  Synthesis  │
   │    Agent    │
   └──────┬──────┘
          ▼
    Final Answer
```

### Quick Setup
1. Run **Cell 1** to install packages
2. **Runtime → Restart runtime**
3. Add `OPENAI_API_KEY` in the 🔑 **Secrets** sidebar
4. Run all remaining cells in order


## Cell 1 — Install packages *(run once, then restart runtime)*

In [ ]:
!pip install -q \
    langgraph==1.2.6 \
    langgraph-checkpoint==4.1.1 \
    langchain==1.3.11 \
    langchain-openai==1.3.3 \
    langchain-community==0.4.2 \
    langchain-core \
    faiss-cpu==1.14.3 \
    tiktoken==0.13.0 \
    openai==1.109.1

print("✅  Installation complete — restart the runtime now (Runtime > Restart runtime)")

## Cell 2 — Load OpenAI API key from Colab Secrets

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("✅  API key loaded")

## Cell 3 — Imports

In [ ]:
import json
import textwrap
from typing import Annotated, Literal

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

print("✅  All imports succeeded")

## Cell 4 — Initialise LLM and Embeddings

In [ ]:
LLM_MODEL   = "gpt-4o-mini"           # all three agents
EMBED_MODEL = "text-embedding-3-small" # FAISS vector store

llm        = ChatOpenAI(model=LLM_MODEL, temperature=0)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print(f"✅  LLM  : {LLM_MODEL}")
print(f"✅  Embed: {EMBED_MODEL}")

## Cell 5 — Sample Corpus

> 💡 Replace or extend `CORPUS` with your own documents.

In [ ]:
CORPUS = [
    # ── Artificial Intelligence ──────────────────────────────────────────
    """Artificial intelligence (AI) refers to the simulation of human intelligence
    in machines programmed to think and learn. Modern AI leverages deep learning,
    transformer architectures, and large-scale datasets. Key sub-fields include
    natural language processing (NLP), computer vision, reinforcement learning,
    and robotics. Foundation models such as GPT-4, Claude, and Gemini are trained
    on internet-scale text and can perform a wide variety of tasks zero-shot.""",

    """Large Language Models (LLMs) are neural networks with billions of parameters
    trained via next-token prediction. Architectures built on the Transformer
    design have become dominant since the 2017 paper 'Attention Is All You Need.'
    LLMs are fine-tuned with RLHF (Reinforcement Learning from Human Feedback)
    to improve instruction-following and safety.""",

    # ── Machine Learning ─────────────────────────────────────────────────
    """Machine learning is a subset of AI focused on algorithms that improve from
    experience. Supervised learning trains on labelled examples; unsupervised
    learning discovers hidden structure; reinforcement learning optimises a policy
    by maximising cumulative reward. Common algorithms include linear/logistic
    regression, decision trees, random forests, gradient boosting, SVMs, and
    neural networks.""",

    """Deep learning uses multi-layer neural networks to learn hierarchical
    representations. CNNs excel at image tasks; RNNs and LSTMs model sequences;
    Transformers use self-attention to model long-range dependencies and now
    dominate NLP, vision, audio, and multimodal benchmarks.""",

    # ── RAG ──────────────────────────────────────────────────────────────
    """Retrieval-Augmented Generation (RAG) enhances LLM outputs by fetching
    relevant documents from an external knowledge base at inference time.
    A typical RAG pipeline: (1) embed the user query, (2) retrieve the top-k
    most similar document chunks via a vector store, (3) prepend those chunks
    as context to the prompt, (4) generate a grounded answer. RAG reduces
    hallucination and allows the model to answer questions beyond its training
    cut-off.""",

    """Advanced RAG variants include HyDE (Hypothetical Document Embeddings),
    multi-query retrieval, step-back prompting, FLARE, Corrective RAG, and
    Self-RAG. Agentic RAG embeds RAG inside an autonomous agent loop that can
    decide when to retrieve, what to retrieve, and can iterate multiple retrieval
    rounds before producing a final answer.""",

    # ── LangGraph ────────────────────────────────────────────────────────
    """LangGraph is a framework for building stateful, multi-actor applications
    with LLMs. It models agent workflows as directed graphs where nodes are Python
    functions and edges encode control flow. State is a typed dict that every node
    reads from and writes to. Conditional edges let the graph route dynamically
    based on LLM decisions, enabling loops, retries, and multi-agent
    collaboration.""",

    """LangGraph supports persistence via checkpointers (in-memory or
    database-backed), allowing long-running agent tasks to be paused and resumed.
    ToolNode automatically executes any tool calls emitted by the LLM and appends
    ToolMessage results back into the conversation state, simplifying the
    tool-use loop.""",

    # ── Vector Databases ─────────────────────────────────────────────────
    """Vector databases store high-dimensional embeddings and support approximate
    nearest-neighbour (ANN) search. Popular options: FAISS (in-process),
    Chroma (local/server), Pinecone (managed cloud), Weaviate (hybrid search),
    Qdrant (Rust-backed), pgvector (PostgreSQL extension). FAISS is commonly
    used in notebooks because it requires no server and supports flat and HNSW
    indices.""",

    # ── Python & Data Science ────────────────────────────────────────────
    """Python is the dominant language for data science and AI. Essential libraries:
    NumPy, Pandas, Matplotlib/Seaborn, Scikit-learn, PyTorch/TensorFlow, and
    Hugging Face Transformers. Jupyter/Colab notebooks provide an interactive
    cell-based environment widely used in research and education.""",

    # ── Climate Change ───────────────────────────────────────────────────
    """Climate change refers to long-term shifts in global temperatures and weather
    patterns. Since the Industrial Revolution, burning fossil fuels, deforestation,
    and industrial agriculture have been the main drivers. Consequences include
    higher average temperatures, more frequent extreme weather events, sea-level
    rise, and ecosystem disruption. The Paris Agreement (2015) targets limiting
    warming to 1.5 degrees C above pre-industrial levels.""",

    # ── Space Exploration ────────────────────────────────────────────────
    """Space exploration accelerated in the 2020s with SpaceX's Starship, NASA's
    Artemis lunar programme, and private Mars missions. The James Webb Space
    Telescope (JWST), launched December 2021, observes in the infrared to study
    the first galaxies, exoplanet atmospheres, and stellar nurseries with
    unprecedented resolution. Commercial launch costs have fallen dramatically.""",
]

print(f"✅  Corpus loaded — {len(CORPUS)} documents")

## Cell 6 — Build FAISS Vector Store

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)

docs = []
for i, text in enumerate(CORPUS):
    chunks = splitter.create_documents(
        [text],
        metadatas=[{"source": f"doc_{i}", "chunk": j}
                   for j in range(len(splitter.split_text(text)))]
    )
    docs.extend(chunks)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✅  Vector store built — {len(docs)} chunks from {len(CORPUS)} documents")

## Cell 7 — Define Tools

| Tool | Purpose |
|---|---|
| `rag_search` | Semantic search over FAISS vector store |
| `calculator` | Safe eval of math expressions |
| `summarise_text` | Condense long text into N sentences |


In [ ]:
@tool
def rag_search(query: str) -> str:
    """
    Search the internal knowledge base using semantic similarity.
    Use whenever the user asks a factual question about AI, ML, RAG,
    LangGraph, vector databases, Python, climate change, or space exploration.
    Returns the most relevant text excerpts.
    """
    results = retriever.invoke(query)
    if not results:
        return "No relevant documents found."
    formatted = []
    for i, doc in enumerate(results, 1):
        src = doc.metadata.get("source", "unknown")
        formatted.append(f"[{i}] (source={src})\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(formatted)


@tool
def calculator(expression: str) -> str:
    """
    Evaluate a simple mathematical expression and return the result.
    Supports +, -, *, /, **, parentheses, abs, round, min, max, pow.
    Example: '(42 * 3.14) / 2'
    """
    try:
        allowed = {
            "__builtins__": {},
            "abs": abs, "round": round, "min": min, "max": max,
            "pow": pow, "sum": sum,
        }
        result = eval(expression, allowed)
        return f"Result: {result}"
    except Exception as exc:
        return f"Calculation error: {exc}"


@tool
def summarise_text(text: str, max_sentences: int = 3) -> str:
    """
    Summarise a long piece of text into at most max_sentences sentences.
    Useful when retrieved documents need to be condensed before inclusion
    in a final answer.
    """
    sentences = [s.strip() for s in text.replace("\n", " ").split(". ") if s.strip()]
    chosen  = sentences[:max_sentences]
    summary = ". ".join(chosen)
    if not summary.endswith("."):
        summary += "."
    return summary


TOOLS      = [rag_search, calculator, summarise_text]
tool_names = {t.name for t in TOOLS}

print("✅  Tools registered:", [t.name for t in TOOLS])

## Cell 8 — Agent State

In [ ]:
class AgentState(TypedDict):
    # add_messages reducer appends new messages (never overwrites)
    messages:     Annotated[list, add_messages]
    # tracks which agent last ran (used by tool router)
    active_agent: str
    # shared notepad — agents write summaries for downstream agents
    scratchpad:   str
    # extracted FINAL_ANSWER string
    final_answer: str

print("✅  AgentState defined")

## Cell 9 — Three Specialised Agents

| Agent | Role | Tools |
|---|---|---|
| **Orchestrator** | Analyses question, routes, may call tools directly | All 3 |
| **Research** | Always retrieves from RAG before answering | `rag_search`, `summarise_text` |
| **Synthesis** | Writes polished final answer from context | None |


In [ ]:
ORCHESTRATOR_SYSTEM = SystemMessage(content="""You are the Orchestrator Agent. Your job is to analyse \
the user's question and decide which specialist agent should handle it.

Rules:
1. If the question involves factual knowledge (AI, ML, RAG, Python, climate, space, etc.),
   call the rag_search tool to gather context first.
2. If the question involves a calculation, call the calculator tool.
3. Once you have sufficient context, write a final answer and end with:
   FINAL_ANSWER: <your answer here>
4. Never fabricate facts — rely entirely on tool results.
5. Keep reasoning concise; use tools efficiently.""")

RESEARCH_SYSTEM = SystemMessage(content="""You are the Research Agent specialised in \
retrieval-augmented generation. You ALWAYS call rag_search to retrieve relevant knowledge
before answering. If retrieved text is too long, call summarise_text to condense it.
After gathering enough context, write a comprehensive grounded answer and end with:
FINAL_ANSWER: <your answer here>""")

SYNTHESIS_SYSTEM = SystemMessage(content="""You are the Synthesis Agent. You receive raw \
retrieved text and reasoning notes from other agents. Craft a clear, well-structured,
human-friendly answer. Do NOT call any tools.
End with:  FINAL_ANSWER: <your answer here>""")

# Bind tools to the agents that need them
orchestrator_llm = ChatOpenAI(model=LLM_MODEL, temperature=0).bind_tools(TOOLS)
research_llm     = ChatOpenAI(model=LLM_MODEL, temperature=0).bind_tools(TOOLS)
synthesis_llm    = ChatOpenAI(model=LLM_MODEL, temperature=0.3)   # no tools

def extract_final_answer(text: str):
    """Pull out text after FINAL_ANSWER: marker, if present."""
    marker = "FINAL_ANSWER:"
    if marker in text:
        return text.split(marker, 1)[1].strip()
    return None

print("✅  Agent LLMs and system prompts ready")

## Cell 10 — Node Functions

In [ ]:
def orchestrator_node(state: AgentState) -> dict:
    """Master coordinator: routes and optionally calls tools."""
    history  = [ORCHESTRATOR_SYSTEM] + state["messages"]
    response = orchestrator_llm.invoke(history)
    final    = extract_final_answer(response.content or "")
    return {
        "messages":     [response],
        "active_agent": "orchestrator",
        "final_answer": final or state.get("final_answer", ""),
        "scratchpad":   state.get("scratchpad", ""),
    }


def research_node(state: AgentState) -> dict:
    """RAG specialist: retrieves then synthesises a grounded answer."""
    history  = [RESEARCH_SYSTEM] + state["messages"]
    response = research_llm.invoke(history)
    final    = extract_final_answer(response.content or "")
    notes    = state.get("scratchpad", "")
    if response.content:
        notes += f"\n[Research] {response.content[:500]}"
    return {
        "messages":     [response],
        "active_agent": "research",
        "final_answer": final or state.get("final_answer", ""),
        "scratchpad":   notes,
    }


def synthesis_node(state: AgentState) -> dict:
    """Polish accumulated context into a clean final answer."""
    scratchpad   = state.get("scratchpad", "")
    last_content = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, (AIMessage, ToolMessage)) and msg.content:
            last_content = msg.content
            break

    orig_question = state["messages"][0].content if state["messages"] else ""
    synthesis_prompt = [
        SYNTHESIS_SYSTEM,
        HumanMessage(content=(
            f"Context gathered by previous agents:\n{scratchpad}\n\n"
            f"Most recent finding:\n{last_content}\n\n"
            f"Original question: {orig_question}\n\n"
            "Write the final answer now."
        )),
    ]
    response = synthesis_llm.invoke(synthesis_prompt)
    final    = extract_final_answer(response.content or "") or response.content
    return {
        "messages":     [response],
        "active_agent": "synthesis",
        "final_answer": final,
        "scratchpad":   state.get("scratchpad", ""),
    }


# Shared ToolNode — handles tool calls from ALL agents
tool_node = ToolNode(TOOLS)

print("✅  Node functions defined")

## Cell 11 — Routing / Edge Functions

In [ ]:
def route_after_orchestrator(state: AgentState) -> Literal[
        "tool_node", "research_node", "synthesis_node", "__end__"]:
    """After orchestrator: execute tools, delegate to research, or finalise."""
    last = state["messages"][-1]
    if state.get("final_answer"):
        return "synthesis_node"
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tool_node"
    content = (last.content or "").lower()
    if any(kw in content for kw in ["retriev", "rag", "knowledge base", "search", "look up"]):
        return "research_node"
    return "synthesis_node"


def route_after_research(state: AgentState) -> Literal["tool_node", "synthesis_node"]:
    """After research agent: run pending tools or move to synthesis."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tool_node"
    return "synthesis_node"


def route_after_tools(state: AgentState) -> Literal["orchestrator_node", "research_node"]:
    """After tools execute: return control to the agent that called them."""
    return "research_node" if state.get("active_agent") == "research" else "orchestrator_node"


def should_end(state: AgentState) -> Literal["__end__", "orchestrator_node"]:
    """After synthesis: always end."""
    return "__end__"

print("✅  Routing functions defined")

## Cell 12 — Build & Compile the LangGraph

In [ ]:
builder = StateGraph(AgentState)

# ── Add nodes ──────────────────────────────────────────────────────────────
builder.add_node("orchestrator_node", orchestrator_node)
builder.add_node("research_node",     research_node)
builder.add_node("synthesis_node",    synthesis_node)
builder.add_node("tool_node",         tool_node)

# ── Entry point ────────────────────────────────────────────────────────────
builder.add_edge(START, "orchestrator_node")

# ── Conditional edges ──────────────────────────────────────────────────────
builder.add_conditional_edges(
    "orchestrator_node",
    route_after_orchestrator,
    {
        "tool_node":      "tool_node",
        "research_node":  "research_node",
        "synthesis_node": "synthesis_node",
        "__end__":        END,
    },
)

builder.add_conditional_edges(
    "research_node",
    route_after_research,
    {
        "tool_node":      "tool_node",
        "synthesis_node": "synthesis_node",
    },
)

builder.add_conditional_edges(
    "tool_node",
    route_after_tools,
    {
        "orchestrator_node": "orchestrator_node",
        "research_node":     "research_node",
    },
)

builder.add_conditional_edges(
    "synthesis_node",
    should_end,
    {"__end__": END, "orchestrator_node": "orchestrator_node"},
)

# ── Compile with in-memory checkpointer (enables multi-turn) ───────────────
memory = MemorySaver()
graph  = builder.compile(checkpointer=memory)

print("✅  LangGraph compiled successfully")

## Cell 13 — Visualise the Graph

In [ ]:
try:
    from IPython.display import Image, display
    img_bytes = graph.get_graph(xray=True).draw_mermaid_png()
    display(Image(img_bytes))
    print("✅  Graph rendered above")
except Exception as e:
    print(f"Graph visualisation skipped: {e}")

## Cell 14 — Run Helper

In [ ]:
def run_agent(question: str, thread_id: str = "default") -> str:
    """
    Run the agentic RAG graph for a question.
    Streams events step-by-step so you can follow every agent/tool call.
    Returns the final synthesised answer string.
    """
    config  = {"configurable": {"thread_id": thread_id}}
    initial = {
        "messages":     [HumanMessage(content=question)],
        "active_agent": "orchestrator",
        "scratchpad":   "",
        "final_answer": "",
    }

    print("\n" + "=" * 65)
    print(f"❓  QUESTION:\n   {question}")
    print("=" * 65)

    final_answer = ""

    for event in graph.stream(initial, config=config):
        for node_name, node_output in event.items():
            msgs = node_output.get("messages", [])
            for msg in msgs:
                preview = ""
                if isinstance(msg, AIMessage):
                    preview = (msg.content or "")[:220]
                    if msg.tool_calls:
                        calls   = [f"{tc['name']}({list(tc['args'].keys())})"
                                   for tc in msg.tool_calls]
                        preview += f"  🔧 TOOL CALLS → {calls}"
                elif isinstance(msg, ToolMessage):
                    preview = f"Tool={msg.name}  →  {(msg.content or '')[:160]}"
                if preview:
                    print(f"\n  [{node_name}] {type(msg).__name__}:")
                    print(f"    {preview.strip()}")
            fa = node_output.get("final_answer", "")
            if fa:
                final_answer = fa

    print("\n" + "─" * 65)
    print("✅  FINAL ANSWER:")
    for line in textwrap.wrap(final_answer, width=63):
        print("   " + line)
    print("─" * 65 + "\n")
    return final_answer

print("✅  run_agent() helper ready")

## Cell 15–19 — Demo Queries

Each query demonstrates a different routing path through the graph.

### Query 1 · RAG — What is RAG?

In [ ]:
run_agent(
    "What is Retrieval-Augmented Generation and how does it reduce hallucination?",
    thread_id="q1"
)

### Query 2 · RAG — LangGraph & ToolNode

In [ ]:
run_agent(
    "Explain how LangGraph models agent workflows and what role ToolNode plays.",
    thread_id="q2"
)

### Query 3 · Multi-hop: RAG + Calculator

In [ ]:
run_agent(
    "If the corpus has 12 documents each split into an average of 3 chunks, how many total chunks are there? Also explain what FAISS is.",
    thread_id="q3"
)

### Query 4 · RAG — Climate Change

In [ ]:
run_agent(
    "What are the main causes and consequences of climate change?",
    thread_id="q4"
)

### Query 5 · Pure Calculation (no RAG)

In [ ]:
run_agent(
    "What is the result of (384 * 1024) / (8 * 1000)?",
    thread_id="q5"
)

## Cell 20 — Ask Your Own Question

In [ ]:
your_question = "What are the differences between supervised and unsupervised learning?"
# ↑ Change this to anything you like!

run_agent(your_question, thread_id="custom")